# Goodreads Machine Learning - Group Project
------------
------------


## Decision/Actions Log : Executif Summary (updated sequentially)
1. clean raw data with 13 columns instead of 12 by concatenating authors column that spilled into average_rating column (fixed manually)
2. clean columns with leading/trailing spaces
3. clean publication_date column (2 errors found, fixed manually with isbn web search)
4. add a column fot publication_year
5. sanitize strings and add a column for clean_title, clean_authors, clean_publisher
6. add a column for the first author and the number of authors


## 0. Import libraries & Helper functions
----------

In [ ]:
import re
import unicodedata
from typing import Any

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
def plot_distribution_with_mean(
    data,
    column,
    *,
    lower=0,
    upper=1500,
    title=None,
    xlabel=None,
    bins=40,
    color="#2a9d8f",
    mean_color="#e76f51",
    figsize=(9, 5),
    show_metrics=True,
):
    """Histogram + KDE with mean line, clipped to [lower, upper]."""
    series = data[column].dropna()
    series = series[(series >= lower) & (series <= upper)]

    col_mean = series.mean()
    col_std = series.std()

    plot_title = title or f"Distribution of {column} ({lower}–{upper})"
    plot_xlabel = xlabel or column

    fig, ax = plt.subplots(figsize=figsize)
    sns.histplot(series, bins=bins, kde=True, color=color, ax=ax)
    ax.axvline(
        col_mean,
        color=mean_color,
        linestyle="--",
        linewidth=2,
        label=f"Mean = {col_mean:.3f}",
    )
    ax.set_xlim(lower, upper)
    ax.set_title(plot_title)
    ax.set_xlabel(plot_xlabel)
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.show()

    if show_metrics:
        display(pd.DataFrame({
            "metric": ["mean", "standard_deviation", "minimum", "maximum", "count"],
            "value": [col_mean, col_std, series.min(), series.max(), len(series)],
        }))

    return fig, ax

In [ ]:
def normalize_string(
    value: Any,
    *,
    lowercase: bool = True,
    strip_accents: bool = True,
    collapse_whitespace: bool = True,
    remove_suffixes: bool = False,
    remove_punctuation: bool = False,
) -> str | None:
    """
    Normalize a single string for fuzzy matching / grouping keys.
    Returns None for None/NaN. Non-strings are cast to str first.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if not s:
        return ""
    if lowercase:
        s = s.lower()
    if strip_accents:
        s = unicodedata.normalize("NFKD", s)
        s = "".join(c for c in s if not unicodedata.combining(c))
    if collapse_whitespace:
        s = re.sub(r"\s+", " ", s)
    if remove_suffixes:
        s = re.sub(
            r"\s+(jr\.?|sr\.?|iii|ii|iv)$",
            "",
            s,
            flags=re.IGNORECASE,
        ).strip()
    if remove_punctuation:
        s = re.sub(r"[^\w\s]", "", s)
        s = re.sub(r"\s+", " ", s).strip()
    return s

--------------------------------------
--------------------------------------
## 1. Loading Data
--------------------------------------
--------------------------------------

In [ ]:
CSV_RAW_PATH = "../data/raw/books.csv"

In [ ]:
df = pd.read_csv(CSV_RAW_PATH, index_col="bookID", on_bad_lines='warn')


Upon inspection, the 4 errors came from the fact that the author string contained a comma "," which was interpreted as a new column and spilled the author name into the nex column when creating the .csv, thus resulting in 13 columns instead of 12

In [ ]:
# the 4 errors were manualy fixed (joining the 2 columns with "," for these 4 error rows)
CSV_CLEANED_PATH = "../data/processed/books-hugo.csv" 
df = pd.read_csv(CSV_CLEANED_PATH, on_bad_lines='warn')

------------------------------------------
------------------------------------------
## 2. EDA & Cleaning Data
------------------------------------------
------------------------------------------

In [ ]:
df.head()

In [ ]:
df.info()

### **observations :**
- It looks like there is a leading space for the " num_pages" column name
- isbn13 as int64 seems like the wrong type for and identifier, but not a big deal because we will probably drop these columns
- publication_date as string is not ideal, should be parsed to date

In [ ]:
# check if bookID, ISBN, ISBN13 are all uniques
cols = ["bookID", "isbn", "isbn13"]
{col: df[col].is_unique for col in cols}

**Set index**

In [ ]:
# since all are unique, we can select any for indexing, lets select bookID
df.set_index("bookID", inplace=True)

**Fix trailing/leading spaces**

In [ ]:
# fix column names
df.columns = df.columns.str.strip()

In [ ]:
# test if any of the values contains leading/trailing spaces
for col in df.columns:
    if df[col].dtype == "str":
        has_spaces = df[col].dropna().ne(df[col].dropna().str.strip()).any()
        if has_spaces:
            print(f"!! Leading/trailing spaces detected in column: {col}")
        else:
            print(f"no spaces detected on column : {col}")



copy dataframe for cleaning

In [ ]:
df_clean = df.copy()

In [ ]:
# fix spaces in titles
df_clean["title"] = df_clean["title"].str.strip()

**Convert publication date to datetime**

In [ ]:
# fix publication_date, passing format to parse
pub_raw = df_clean["publication_date"].copy()
df_clean["publication_date"] = pd.to_datetime(
    df_clean["publication_date"],
    format="%m/%d/%Y",
    errors="coerce", # if the format is not correct, set to NaT
)
# check original values where the conversion failed
pub_raw.loc[df_clean["publication_date"].isna()]

2 dates did not convert properly : these dates are impossible, lets check ISBN code and look for the information on the internet since there is only 2 books with invalid dates

In [ ]:
# check ISBN13 code
df_clean.loc[df_clean.publication_date.isna(), ["isbn", "isbn13", "title"]]

According to current goodreads.com: 
- *In Pursuit of the Proper Sinner* was puplished in October 31st, 2000
- *Montaillou  village occitan de 1294 à 1324* was published in June 30, 1982

One date is wrong by 1 day, the other by 1 month. 

These errors could be isolated, or might be a symptom of a larger issue with dates. Some of the parsable date might still be wrong. Goodreads.com does not provide a public API to check the data programmatically (an unofficial one exist on Apify), but there are other options to check the publishing date with Open Library or Google Books API.  

Lets apply a target fix:

In [ ]:
# fix the dates by index
df_clean.loc[31373, "publication_date"] = pd.to_datetime("2000-10-31")
df_clean.loc[45531, "publication_date"] = pd.to_datetime("1982-06-30")

Lets take the opportunity to add a new column for year of publication as it will be more usefull that the exact date

In [ ]:
df_clean["publication_year"] = df_clean["publication_date"].dt.year

**String Sanitization : apply normalization function**

In [ ]:
df_clean["title"] = df_clean["title"].apply(normalize_string)
df_clean["authors"] = df_clean["authors"].apply(normalize_string)
df_clean["publisher"] = df_clean["publisher"].apply(normalize_string)

**Creating new colums for *Authors* : inspection of the data show that authors are separated by "/"**

In [ ]:
# check what is the maximum number of authors for one book
n_authors = df_clean["authors"].str.split("/").str.len()
print("max number of authors for one book :", n_authors.max())

51 authors for one book !

In [ ]:
cols = ["title", "authors"]
df_clean.loc[n_authors == n_authors.max(), cols]

To avoid empty data, and because the first author is usually the main one (confirmed below), lets add a column for the first author only, and one column with the number of authors

In [ ]:
# creating new columns
df_clean["first_author"] = df_clean["authors"].str.split(
    "/").str[0].str.strip()
df_clean["n_authors"] = n_authors

In [ ]:
# plot showing the number of authors per book
plt.figure(figsize=(10, 6))
sns.histplot(df_clean["n_authors"], bins=range(1, n_authors.max() + 2))
plt.title("Distribution of Number of Authors per Book")
plt.xlabel("Number of Authors")
plt.ylabel("Count")
plt.show()

It looks like books with more than 10 authors are rare, we might want to remove them from the dataset or use a logaritmic scale later on.

In [ ]:
# Authors per book — most books have one author; tail is long
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.countplot(
    data=df_clean.assign(author_bucket=df_clean["n_authors"].clip(upper=10).astype(int)),
    x="author_bucket",
    color="#2a9d8f",
    ax=axes[0],
)
axes[0].set_title("Authors per book (capped at 10 for readability)")
axes[0].set_xlabel("Number of authors")
axes[0].set_ylabel("Books")

high_author_share = (df_clean["n_authors"] > 10).mean()
axes[0].text(
    0.98,
    0.95,
    f"{high_author_share:.2%} of rows have >10 authors",
    transform=axes[0].transAxes,
    ha="right",
    va="top",
    fontsize=9,
)

sns.boxplot(data=df_clean, x="n_authors", y="average_rating",
            color="#e9c46a", ax=axes[1])
axes[1].set_title("Rating vs author count (raw counts)")
axes[1].set_xlabel("Number of authors")
plt.tight_layout()
plt.show()

## Looking for blank/empty/null/bad data


In [ ]:
df_clean.describe()

At first glance, we can see that there are rows with num_pages and text_reviews_count equal to 0  
➜ should be investigated and cleaned

In [ ]:
df_clean.isna().sum()

Check for other bad/empty data :

In [ ]:
# check for bad data
bad_inputs = ["", "nan", "None", "N/A", "null", "-"]
bad_results = {}
for col in df.columns:
    mask = df_clean[col].astype(str).str.strip().str.lower().isin(bad_inputs)
    bad_results[col] = int(mask.sum())
bad_results

➜ no strictly missing values
But visual analysis show that multiple entries have "NOT A BOOK" as authors

In [ ]:
# count the number of books with "NOT A BOOK" as authors
print("number of books with 'NOT A BOOK' as authors :", (df_clean["authors"] == "NOT A BOOK".lower()).sum())

# drop these rows
#df_clean = df_clean[df_clean["authors"] != "NOT A BOOK"]

In [ ]:
df_clean.loc[df_clean["authors"] == "NOT A BOOK".lower(),:]

This looks like audio books or podcasts

# Zero values

In [ ]:
print("num_pages zero count :", (df_clean["num_pages"] == 0).sum())
print("ratings_count zero count :", (df_clean["ratings_count"] == 0).sum())
print("text_reviews_count zero count :", (df_clean["text_reviews_count"] == 0).sum())

In [ ]:
zero_pages = df_clean.loc[df_clean["num_pages"] == 0, "average_rating"]
weights = np.ones(len(zero_pages)) / len(zero_pages) * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(zero_pages, bins=20, weights=weights,
        color="#2a9d8f", edgecolor="white")
ax.set_title(f"Average rating — 0-page books (n={len(zero_pages)})")
ax.set_xlabel("average_rating")
ax.set_ylabel("% of 0-page books")
plt.tight_layout()
plt.show()

In [ ]:
zero_pages = df_clean.loc[df_clean["num_pages"] == 0, "average_rating"]

bins = [0, 1, 2, 3, 4, 4.5, 5.01]
labels = ["0–1", "1–2", "2–3", "3–4", "4–4.5", "4.5–5"]


def rating_dist_pct(series):
    return (
        pd.cut(series, bins=bins, labels=labels, right=False)
        .value_counts(normalize=True)
        .reindex(labels)  # keep all bins, even 0%
        .mul(100)
        .round(2)
    )


compare = pd.DataFrame({
    "All books": rating_dist_pct(df_clean["average_rating"]),
    "0 pages": rating_dist_pct(zero_pages),
})

compare

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

compare.plot(
    kind="bar",
    ax=ax,
    width=0.8,
    color=["#2a9d8f", "#e76f51"],
    edgecolor="black",
    linewidth=0.5,
)

ax.set_title("Average rating distribution (%): all books vs 0-page books")
ax.set_xlabel("Average rating bin")
ax.set_ylabel("% of books in group")
ax.legend(title="")
ax.set_xticklabels(labels, rotation=0)

# optional: show n in legend or subtitle
n_all = len(df_clean)
n_zero = len(zero_pages)
ax.text(
    0.99, 0.99,
    f"n(all)={n_all:,}  |  n(0 pages)={n_zero}",
    transform=ax.transAxes,
    ha="right", va="top",
    fontsize=9,
)

plt.tight_layout()
plt.show()

In [ ]:
zero_reviews = df_clean.loc[df_clean["text_reviews_count"]
                            == 0, "average_rating"]

bins = [0, 1, 2, 3, 4, 4.5, 5.01]
labels = ["0–1", "1–2", "2–3", "3–4", "4–4.5", "4.5–5"]


def rating_dist_pct(series):
    return (
        pd.cut(series, bins=bins, labels=labels, right=False)
        .value_counts(normalize=True)
        .reindex(labels)
        .mul(100)
        .round(2)
    )


compare_reviews = pd.DataFrame({
    "All books": rating_dist_pct(df_clean["average_rating"]),
    "0 text reviews": rating_dist_pct(zero_reviews),
})

compare_reviews

In [ ]:
n_all = len(df_clean)
n_zero_rev = len(zero_reviews)

fig, ax = plt.subplots(figsize=(10, 5))

compare_reviews.plot(
    kind="bar",
    ax=ax,
    width=0.8,
    color=["#2a9d8f", "#e76f51"],
    edgecolor="black",
    linewidth=0.5,
)

ax.set_title("Average rating distribution (%): all books vs 0 text reviews")
ax.set_xlabel("Average rating bin")
ax.set_ylabel("% of books in group")
ax.legend(title="")
ax.set_xticklabels(labels, rotation=0)
ax.text(
    0.99, 0.99,
    f"n(all)={n_all:,}  |  n(0 text reviews)={n_zero_rev}",
    transform=ax.transAxes,
    ha="right", va="top",
    fontsize=9,
)

plt.tight_layout()
plt.show()

**What to do with these : further analysis needed**

num_pages = 0 ➜ encode average/median?  
ratings_count = 0 ➜ drop ?
text_reviews_count = 0 ➜ could be an actual signal, keep ?

In [ ]:
# drop rows with ratings_count = 0
df_clean = df_clean[df_clean["ratings_count"] > 0]

# Feature Engineering : collections, series, audiobooks
Some titles either multiple books from a collection, or are one volume that is part of a collection, this is usually indicated by a parenthetical note after the title.

In [ ]:
# Adding a boolean column to identify wether or not a record a collection, part of a series or an audio book. 
# The identifiers for series are "#[number]" / "Volume [number]" / "Vol. [number]" / "Books [number]" / "Part [number] /"Tome [number]"
REGEX_FLAGS = re.IGNORECASE | re.VERBOSE

NUM = r"(?:\d+(?:\.\d+)?|[ivxlcdm]+|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)"
NUM_RANGE = rf"{NUM}(?:\s*(?:[-\u2013\u2014,/&]|and)\s*{NUM})*"

collection_pattern = r"""
\b(?:box(?:ed)?\s+set|omnibus|antholog(?:y|ies)|collection)\b
|
\b(?:complete|collected|selected|essential|major)\s+
(?:works|novels|stories|short\s+stories|poems|poetry|plays|writings|letters|essays|tragedies|adventures|memoirs)\b
|
\b\d+\s*(?:book|volume)s?\s+(?:box(?:ed)?\s+set|set|collection)\b
|
\b(?:\d+\s+)?volumes?\s+set\b
"""

series_pattern = rf"""
\([^)]*(?:\#\s*{NUM_RANGE}|\b(?:vol(?:ume)?s?\.?|books?|part|tome)\s+{NUM_RANGE})[^)]*\)
|
(?<!\w)\#\s*{NUM_RANGE}\b
|
\b(?:vol(?:ume)?s?\.?|part|tome)\s+{NUM_RANGE}\b
"""

# Optional broad add-on, but noisier:
book_number_pattern = rf"\bbooks?\s+{NUM_RANGE}\b"

audio_pattern = r"""
audio(?:books?|go|text)?\b
|
\bcassettes?\b
|
\bcd\s*audio\b
|
\baudio\s*cd\b
|
\blistening\s+library\b
"""

title = df_clean["title"].fillna("")
title_publisher = (
    df_clean["title"].fillna("") + " " + df_clean["publisher"].fillna("")
)

df_clean["is_collection"] = title.str.contains(
    collection_pattern, regex=True, flags=REGEX_FLAGS, na=False)
df_clean["is_series"] = title.str.contains(
    series_pattern, regex=True, flags=REGEX_FLAGS, na=False)
df_clean["is_audio"] = title_publisher.str.contains(
    audio_pattern, regex=True, flags=REGEX_FLAGS, na=False)

In [ ]:
# graph to show the number of books that are collection, series and audio (in the same graph)
identifier_counts = (
    df_clean[["is_collection", "is_series", "is_audio"]]
    .sum()
    .rename({
        "is_collection": "Collections",
        "is_series": "Part of series",
        "is_audio": "Audiobooks",
    })
)

ax = identifier_counts.plot(
    kind="bar",
    figsize=(7, 4),
    color=["#4C78A8", "#F58518", "#54A24B"],
)

ax.set_title("Count of Identifier Categories")
ax.set_ylabel("Number of books")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()

In [ ]:
rating_means = pd.Series({
    "Collections": df_clean.loc[df_clean["is_collection"], "average_rating"].mean(),
    "Part of series": df_clean.loc[df_clean["is_series"], "average_rating"].mean(),
    "Audiobooks": df_clean.loc[df_clean["is_audio"], "average_rating"].mean(),
})

ax = rating_means.plot(
    kind="bar",
    figsize=(7, 4),
    color=["#4C78A8", "#F58518", "#54A24B"],
)

ax.set_title("Average Rating by Identifier Category")
ax.set_ylabel("Average rating")
ax.set_xlabel("")
ax.set_ylim(0, 5)
ax.tick_params(axis="x", rotation=0)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f")

plt.tight_layout()
plt.show()

In [ ]:
audio_ratings = df_clean.loc[
    df_clean["is_audio"], "average_rating"
].dropna()

plt.figure(figsize=(8, 4))

sns.histplot(
    audio_ratings,
    bins=20,
    kde=True,
    color="#54A24B",
    edgecolor="white"
)

plt.axvline(
    audio_ratings.mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {audio_ratings.mean():.2f}"
)

plt.title("Average Rating Distribution for Audiobooks")
plt.xlabel("Average rating")
plt.ylabel("Number of audiobooks")
plt.xlim(0, 5)
plt.legend()
plt.tight_layout()
plt.show()

## Looking for silent duplicates
------------------------

We already established that there are no strict duplicates, since bookID, isbn and isbn13 are all uniques

**Title duplicates**

In [ ]:
# number of duplicated titles
print("number of duplicated titles :",(df_clean["title"].value_counts() > 1).sum())

The clean "title" is a sanitized version of the title, it is more reliable to use it for duplicate detection but we need to check if there are some cases where the cleaned title is the same but the raw title is actually different on purpose.



There are a few duplicated titles, but they might have different authors

In [ ]:
#  duplicated titles & authors
dup_counts = df_clean[["title", "authors"]].value_counts()
print("number of duplicated titles & authors :",(dup_counts > 1).sum())
# (df[["title","authors"]].value_counts()>1).sum()

In [ ]:
# check if there exist books with same title and authors, but different ratings
print("number of books with same title and authors, but different ratings :", (df_clean.groupby(["title","authors"])["average_rating"].nunique() > 1).sum())

In [ ]:
# check if there exist books with same title and authors, but different ratings
print("number of books with same title and authors, but different ratings :", (df_clean.groupby(["title", "authors"])["average_rating"].nunique() > 1).sum())

Good news, duplicated books (by title & authors) all have the same ratings

In [ ]:
# number of book with same title but different authors
df_same_title_not_authors = df_clean.groupby(
    ["title"])["authors"].nunique() > 1
print("number of books with same title but different authors :",df_same_title_not_authors.sum())

In [ ]:
df_clean["title"].value_counts()

In [ ]:
df_clean[df_clean["title"] == "the iliad"][:]

From this example, we can infer that even though the authors are differents, the first author is the same.
We might use this information to merge the books with the same title but different authors.

In [ ]:
# number for books with same title but different first author
df_same_title_not_1st_author = df_clean.groupby(
    ["title"])["first_author"].nunique() > 1
print("number of books with same title but different first author :",df_same_title_not_1st_author.sum())

In [ ]:
print("number of books with same title and first_author but different average rating :",
      (df_clean.groupby(["title", "first_author"])
       ["average_rating"].nunique() > 1).sum()
)

In [ ]:
# Show the books with same title and first_author but different average rating
(
    df_clean.groupby(["title", "first_author"])["average_rating"]
    .agg(n_unique_average_rating="nunique", ratings_list=lambda x: list(x.unique()))
    .reset_index()
    .query("n_unique_average_rating > 1")
    .head(10)
)

In [ ]:

(
    df_clean.groupby(["title", "authors"], as_index=False)
    .agg(
        n_copies=("title", "size"),
        language_code=("language_code", list),
        num_pages=("num_pages", list),
        publisher=("publisher", list),
        publication_year=("publication_year", list),
        average_rating=("average_rating", list),
        ratings_count=("ratings_count", list),
        text_reviews_count=("text_reviews_count", list),
    )
    .query("n_copies > 1")
    .nlargest(20, "n_copies")
)

As seen earlier, there are no truly duplicated rows, but some of the rows only differ by publisher or language code : average_rating is the same when title & authors are the same.

## Analysis : Average Rating (Target)
-----------

In [ ]:
plot_distribution_with_mean(
    df_clean,
    column="average_rating",
    lower=0,
    upper=df_clean["average_rating"].max(),
    title="Average Rating — Distribution Around the Mean",
    xlabel="Average Rating",
)

**Interpretation**  
- distribution is heavily centered around the average : 3.93  
- a few books appear with average rating being zero

In [ ]:
print("average_rating negative count :",(df_clean["average_rating"] < 0).sum())

## Analysis : Ratings Count
-----------

In [ ]:
# correlation between average_rating and ratings_count
sns.regplot(x="average_rating", y="ratings_count", data=df_clean)
plt.show()

In [ ]:
support_bins = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 25, 100, 1_000, 10_000, np.inf]
support_labels = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10-25", "26-100", "101-1k", "1k-10k", "10k+"]
df_clean["rating_support_band"] = pd.cut(
    df_clean["ratings_count"],
    bins=support_bins,
    labels=support_labels,
    include_lowest=True,
)

support_table = (
    df_clean.groupby("rating_support_band", observed=True)
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        median_ratings_count=("ratings_count", "median"),
    )
    .reset_index()
)
support_table["share_of_df_clean"] = support_table["books"] / len(df_clean)
display(support_table)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(
    data=support_table,
    x="rating_support_band",
    y="books",
    color="#2a9d8f",
    ax=axes[0],
)
axes[0].set_title("Books by rating-support band")
axes[0].set_xlabel("Ratings count band")
axes[0].set_ylabel("Number of books")
axes[0].tick_params(axis="x", rotation=30)

sns.lineplot(
    data=support_table,
    x="rating_support_band",
    y="rating_std",
    marker="o",
    color="#e76f51",
    ax=axes[1],
)
axes[1].set_title("Rating spread is wider for low-support books")
axes[1].set_xlabel("Ratings count band")
axes[1].set_ylabel("Std. dev. of average_rating")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## Analysis : Number of pages
-----------

Lets decide arbitrarily how many pages a book needs to have to be considered a real book

In [ ]:
MIN_PAGES_COUNT = 5
MAX_PAGES_COUNT = 3000

In [ ]:
print(
    f'books with less than {MIN_PAGES_COUNT} pages : {(df_clean["num_pages"]<MIN_PAGES_COUNT).sum()}\n'
    f'books with more than {MAX_PAGES_COUNT} pages : {(df_clean["num_pages"]>MAX_PAGES_COUNT).sum()}\n'
    f'books with zero pages : {(df_clean["num_pages"] == 0).sum()}'
)
 

In [ ]:
df_clean.loc[df_clean["num_pages"]>MAX_PAGES_COUNT,["title","num_pages", "average_rating"]]

In [ ]:
# num_pages
plot_distribution_with_mean(
    df_clean,
    column="num_pages",
    lower=0,
    upper=df_clean["num_pages"].max(),
    title="Number of Pages — Distribution Around the Mean",
    xlabel="Number of pages",
)

## Analysis : Language Codes
------------

In [ ]:
language_value_counts = df_clean["language_code"].value_counts(dropna=False)
language_value_counts

In [ ]:
lang_values = language_value_counts.index.to_list()
lang_values

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

language_value_counts = df_clean["language_code"].value_counts(dropna=False)

language_plot = df_clean[["language_code", "average_rating"]].copy()

order = (
    language_plot.groupby("language_code", dropna=False)["average_rating"]
    .mean()
    .sort_values(ascending=False)
    .index
)

sns.pointplot(
    data=language_plot,
    x="language_code",
    y="average_rating",
    order=order,
    errorbar=("ci", 95),
    color="#2a9d8f",
    ax=ax,
)
ax.set_title("Language & ratings")
ax.set_xlabel("Language Code")
ax.set_ylabel("Mean average rating")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# check for multi-language books
multi_lang = (
    df_clean.groupby(["title", "authors"])["language_code"]
    .agg(n_languages="nunique", languages=lambda s: sorted(s.dropna().unique()))
    .reset_index()
    .query("n_languages > 1")
    .sort_values("n_languages", ascending=False)
    .head(50)
)
multi_lang